In [ ]:
#imports
import nibabel as nib
#import nilearn
import numpy as np
import os
import cv2

import cc3d
from matplotlib import pyplot as plt

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

#from tqdm import tqdm
from skimage.io import imread, imshow
from skimage.transform import resize

import tensorflow as tf

import keras.backend as K
from keras.models import Model, load_model
from keras.layers import Input, BatchNormalization, Activation, Dense, Dropout
from keras.layers.core import Lambda, RepeatVector, Reshape
from keras.layers.convolutional import Conv2D, Conv2DTranspose
from keras.layers.pooling import MaxPooling2D, GlobalMaxPool2D
from tensorflow.keras.layers import concatenate
#from keras.layers.merge import concatenate, add
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.callbacks import CSVLogger
#from keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array, load_img
from tensorflow.keras.models import *
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, TensorBoard
from tensorflow.keras.layers.experimental import preprocessing

from tensorflow.keras.losses import binary_crossentropy

import tensorflow.keras.layers as L
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Conv2D, Conv1D
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, Input
from tensorflow.keras.layers import Activation, Concatenate, Conv2D, Multiply, Add

In [ ]:
# SE module
import tensorflow as tf
import tensorflow.keras.layers as L
from tensorflow.keras.models import Model

def SE(inputs, ratio=8):
    ## [8, H, W, 32]
    channel_axis = -1
    num_filters = inputs.shape[channel_axis]
    se_shape = (1, 1, num_filters)

    x = L.GlobalAveragePooling2D()(inputs) ## [8, 32]
    x = L.Reshape(se_shape)(x)
    x = L.Dense(num_filters // ratio, activation='relu', use_bias=False)(x)
    x = L.Dense(num_filters, activation='sigmoid', use_bias=False)(x)

    x = L.Multiply()([inputs, x])
    return x

#cbam
import tensorflow as tf
from tensorflow.keras.layers import GlobalAveragePooling2D, GlobalMaxPooling2D, Reshape, Dense, Input
from tensorflow.keras.layers import Activation, Concatenate, Conv2D, Multiply, Add

def channel_attention_module(x, ratio=8):
    channel = x.shape[-1]

    ## Shared layers
    l1 = Dense(channel // ratio, activation="relu", use_bias=False)
    l2 = Dense(channel, use_bias=False)

    ## Global Average Pooling
    x1 = GlobalAveragePooling2D()(x)
    x1 = l1(x1)
    x1 = l2(x1)

    ## Global Max Pooling
    x2 = GlobalMaxPooling2D()(x)
    x2 = l1(x2)
    x2 = l2(x2)

    ## Add both the features and pass through sigmoid
    feats = x1 + x2
    feats = Activation("sigmoid")(feats)
    feats = Multiply()([x, feats])

    return feats

def spatial_attention_module(x):
    ## Average Pooling
    x1 = tf.reduce_mean(x, axis=-1, keepdims=True)

    ## Max Pooling
    x2 = tf.reduce_max(x, axis=-1, keepdims=True)

    ## Concatenate both the features
    feats = Concatenate()([x1, x2])
    ## Conv layer
    feats = Conv2D(1, kernel_size=7, padding="same", activation="sigmoid")(feats)
    feats = Multiply()([x, feats])

    return feats


def cbam(x):
    x = channel_attention_module(x)
    x = spatial_attention_module(x)
    return x

def Global_self_attention_block(x, filters):
    # Compute query, key, value
    print('x=', x.shape)
    query = Conv2D(filters/2, kernel_size=1, activation='relu')(x)
    print('q=',query.shape)
    query = GlobalAveragePooling2D(keepdims=True)(query)
    print('gq',query.shape)    
    #query = tf.expand_dims(query, axis=-2)
    #print('gq_ed',query.shape)
    
    key = Conv2D(filters/2, kernel_size=1, activation='relu')(x)
    print('k=',key.shape)
    
    value = Conv2D(filters/2, kernel_size=1, activation='relu')(x)
    print('v=',value.shape)
    
    # Reshape and compute attention scores
    query_shape = tf.shape(key)
    #query = tf.reshape(query, (query_shape[0], -1, query_shape[-1]))  # (batch, height*width, filters//8)
    key = tf.reshape(key, (query_shape[0],query_shape[1] * query_shape[2],query_shape[-1]))  # (batch, height*width, filters//8)
    print('r_k=',key.shape)
    key_t = tf.transpose(key,perm=[0, 2, 1])
    print('key_t',key_t.shape)
    
    value = tf.reshape(value, (query_shape[0],query_shape[1] * query_shape[2], query_shape[-1]))  # (batch, height*width, filters)
    print('r_v=',value.shape)

    #qk_attention_scores = tf.matmul(query, key_t)# transpose_b=True)  # (batch, height*width, height*width)
    qk_attention_scores = K.batch_dot(query, key_t)
    print('qxk=', qk_attention_scores.shape)
    attention_scores = Softmax()(qk_attention_scores)
    #attention_scores_shape = tf.shape(attention_scores)

    # Compute attention-weighted value
    #qkv_attention_output = tf.matmul(attention_scores, value)  # (batch, height*width, filters)
    qkv_attention_output =  K.batch_dot(attention_scores, value)
    print('qkXv=',qkv_attention_output.shape)
    
    
    qk = tf.reshape(attention_scores,(query_shape[0],query_shape[1],query_shape[2]))
    print('re_qk=', qk.shape)
    qk = tf.expand_dims(qk, axis=-1)
    print('re_qk_ed=', qk.shape)
    #fsp = x * qk
    fsp = Multiply()([x, qk])
    print('fsp=',fsp.shape)
    
    f = Conv1D(filters, kernel_size=1, activation="sigmoid")(qkv_attention_output)
    #print('fc',fc.shape)
    #f = L.Dense(filters, activation='sigmoid')(qkv_attention_output)
    #f = L.Dense(filters, use_bias=False)(f)
    print('f',f.shape)
    #fch = x * f
    fch = Multiply()([x,f])
    print('fch',fch.shape)
    
    # Define trainable scalar parameters
    #W1 = tf.Variable(0.5, name='W1', trainable=True)
    #W2 = tf.Variable(1.0, name='W2', trainable=True)
    
    #fgs = fsp + fch
    fgs = tf.add(fsp,fch)
    print('fgs',fgs.shape)
    
    #skip = Multiply()([x,fgs])
    #print('skip=',skip.shape)
    
    #fgs = tf.concat([fsp, fch], axis=1

    return fgs

def spatial_attention_module(x):
    ## Average Pooling
    x1 = tf.reduce_mean(x, axis=-1)
    x1 = tf.expand_dims(x1, axis=-1)

    ## Max Pooling
    x2 = tf.reduce_max(x, axis=-1)
    x2 = tf.expand_dims(x2, axis=-1)

    ## Concatenat both the features
    feats = Concatenate()([x1, x2])
    print('conc',feats.shape)
    ## Conv layer
    feats = Conv2D(1, kernel_size=7, padding="same", activation="sigmoid")(feats)
    feats = Multiply()([x, feats])
    print('mult',feats.shape)

    return feats

def spatial_attention_module_K3(x):
    ## Average Pooling
    x1 = tf.reduce_mean(x, axis=-1)
    x1 = tf.expand_dims(x1, axis=-1)

    ## Max Pooling
    x2 = tf.reduce_max(x, axis=-1)
    x2 = tf.expand_dims(x2, axis=-1)

    ## Concatenat both the features
    feats = Concatenate()([x1, x2])
    print('conc',feats.shape)
    ## Conv layer
    feats = Conv2D(1, kernel_size=3, padding="same", activation="sigmoid")(feats)
    feats = Multiply()([x, feats])
    print('mult',feats.shape)

    return feats

def spatial_attention_module_K5(x):
    ## Average Pooling
    x1 = tf.reduce_mean(x, axis=-1)
    x1 = tf.expand_dims(x1, axis=-1)

    ## Max Pooling
    x2 = tf.reduce_max(x, axis=-1)
    x2 = tf.expand_dims(x2, axis=-1)

    ## Concatenat both the features
    feats = Concatenate()([x1, x2])
    print('conc',feats.shape)
    ## Conv layer
    feats = Conv2D(1, kernel_size=5, padding="same", activation="sigmoid")(feats)
    feats = Multiply()([x, feats])
    print('mult',feats.shape)

    return feats

def triplet_attention(inputs,filters):
    print('input',inputs.shape)
    #x_perm1 = tf.transpose(inputs, perm=[0, 2, 1, 3])
    #print('trans1',x_perm1.shape)
    #x_out1 = spatial_gate(x_perm1)
    #x_out1 = spatial_attention_module(x_perm1)
    #x_out11 = tf.transpose(x_out1, perm=[0, 2, 1, 3])
    #print('wh',x_out11.shape)
    
    x_out_K3 = spatial_attention_module_K3(inputs)   #HW
    x_out_K5 = spatial_attention_module_K5(inputs)
    x_out_k3k5 =x_out_K3 + x_out_K5
    
    
    x_perm2 = tf.transpose(inputs, perm=[0, 3, 2, 1])  #CW
    print('trans2',x_perm2.shape)
    x_out2 = spatial_attention_module(x_perm2)
    #x_out2 = spatial_gate(x_perm2)
    x_out21 = tf.transpose(x_out2, perm=[0, 3, 2, 1])
    print('CW',x_out21.shape)
    
    x_perm3 = tf.transpose(inputs, perm=[0, 1, 3, 2])   #HC
    print('trans3',x_perm3.shape)
    x_out3 = spatial_attention_module(x_perm3)
    #x_out2 = spatial_gate(x_perm2)
    x_out31 = tf.transpose(x_out3, perm=[0, 1, 3, 2])
    print('CH',x_out31.shape)
    
    #x_out = spatial_gate(inputs)
    x_outg = Global_self_attention_block(inputs,filters)  #GSA
    
    #x_out = (1 / 4) * (x_out + x_out11 + x_out21 + x_out31)
    x_out = (1 / 4) * (x_outg + x_out21 + x_out31 + x_out_k3k5)
        
    return x_out

In [ ]:
from tensorflow.keras import layers, models, Model
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, MaxPooling2D, concatenate, Input, BatchNormalization, Activation

def unet_block(input_tensor, num_filters):
    x = Conv2D(num_filters, (3, 3), padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

def Res_unet_block(input_tensor, num_filters):
    x = Conv2D(num_filters, (3, 3), padding="same")(input_tensor)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, (3, 3), padding="same")(x)
    x = BatchNormalization()(x)
    # Residual/Skip connection
    res = Conv2D(num_filters, kernel_size=1, padding='same')(input_tensor)
    x = Add()([res, x])
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def attention_block(l_layer,h_layer): #Attention Block
    phi=Conv2D(h_layer.shape[-1],(1,1),padding='same')(l_layer)
    theta=Conv2D(h_layer.shape[-1],(1,1),strides=(2,2),padding='same')(h_layer)
    x=tf.keras.layers.add([phi,theta])
    x=Activation('relu')(x)
    x=Conv2D(1,(1,1),padding='same',activation='sigmoid')(x)
    x=UpSampling2D(size=(2,2))(x)
    x=tf.keras.layers.multiply([h_layer,x])
    x=BatchNormalization(axis=3)(x)
    return x

def SLGA_Net(input_shape=(128, 128, 1), num_classes=1):
    inputs = Input(input_shape)
    features = 32

    # Encoder
    enc1 = unet_block(inputs, features)
    pool1 = MaxPooling2D(pool_size=(2, 2))(enc1)
    
    enc2 = unet_block(pool1, features * 2)
    enc2_tr = triplet_attention(enc2,features * 2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(enc2_tr)
    
    enc3 = unet_block(pool2, features * 4)
    enc3_tr = triplet_attention(enc3,features * 4)
    pool3 = MaxPooling2D(pool_size=(2, 2))(enc3_tr)
    
    enc4 = unet_block(pool3, features * 8)
    enc4_tr = triplet_attention(enc4,features * 8)
    pool4 = MaxPooling2D(pool_size=(2, 2))(enc4_tr)

    # Bottleneck
    bottleneck = unet_block(pool4, features * 16)
    bottleneck = cbam(bottleneck)

    # Decoder
    upconv4 = Conv2DTranspose(features * 8, (2, 2), strides=(2, 2), padding="same")(bottleneck)
    dec4 = concatenate([upconv4, enc4])
    dec4 = unet_block(dec4, features * 8)
    dec4 = triplet_attention(dec4, features * 8)   #Decoder CTAM
 
    
    upconv3 = Conv2DTranspose(features * 4, (2, 2), strides=(2, 2), padding="same")(dec4)#(enc4)#(dec4)
    dec3 = concatenate([upconv3, enc3])
    dec3 = unet_block(dec3, features * 4)
    dec3 = triplet_attention(dec3,features * 4)

    
    upconv2 = Conv2DTranspose(features * 2, (2, 2), strides=(2, 2), padding="same")(dec3)
    dec2 = concatenate([upconv2, enc2]) 
    dec2 = unet_block(dec2, features * 2)
    dec2 = triplet_attention(dec2,features * 2)
    
    upconv1 = Conv2DTranspose(features, (2, 2), strides=(2, 2), padding="same")(dec2)
    dec1 = concatenate([upconv1, enc1])
    dec1 = unet_block(dec1, features)

    outputs = Conv2D(num_classes, (1, 1), activation="sigmoid")(dec1)

    return Model(inputs, outputs)

# Create the model
model = SLGA_Net(input_shape=(128, 128, 1), num_classes=1)
model.summary()
